<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/Module1_Labs(v2)/Lab1_Python_Colab_Qiskit_Quickstart.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# QOS Lab 1 — Python, Colab & Qiskit Quickstart
### Quantum Optimization and Simulation | Cleveland State University
**Instructor:** Prof. Chansu Yu | Washkewicz College of Engineering

---

## Learning Objectives
By the end of this lab you will be able to:
1. Navigate a Jupyter/Colab notebook (cells, running, restarting)
2. Write basic Python: variables, lists, loops, functions
3. Use NumPy for matrix and vector operations (essential for quantum gates)
4. Install and import Qiskit, and run your first quantum circuit
5. Measure a qubit and interpret the result

---

## 🗺️ Survival Guide: Jupyter / Google Colab in 60 Seconds

| Action | Keyboard Shortcut |
|--------|------------------|
| **Run** current cell | `Shift + Enter` |
| **Run** and insert new cell below | `Alt + Enter` |
| Add cell **below** | `B` (in command mode) |
| Add cell **above** | `A` (in command mode) |
| Switch to **command mode** | `Esc` |
| Switch to **edit mode** | `Enter` |
| **Restart** kernel (clear all variables) | Runtime → Restart runtime |

> **Tip:** Cells run in order. If you skip a cell, variables defined in it won't exist. When in doubt, use **Runtime → Run all**.

---

## Part 1: Python Basics — A Quick Tour for C++/Java Programmers

Python syntax is lighter than C++ or Java. Key differences:
- **No semicolons** at line ends
- **No type declarations** — Python infers types
- **Indentation** (4 spaces) replaces `{ }` for code blocks
- **No `main()`** — code runs top to bottom

In [ ]:
# ── 1.1  Variables and types ─────────────────────────────────────────────────
# In C++: int x = 5;  float pi = 3.14;  string s = "hello";
# In Python:
x   = 5
pi  = 3.14159
msg = "hello, quantum world"

print(type(x),   x)
print(type(pi),  pi)
print(type(msg), msg)

In [ ]:
# ── 1.2  Lists (like arrays, but resizable and mixed-type) ───────────────────
nodes   = [0, 1, 2, 3, 4]          # 5-node graph (we'll use this later!)
labels  = ['A', 'B', 'C']
mixed   = [1, 3.14, 'qubit', True]

print(nodes)
print("First element:",  nodes[0])  # 0-indexed, same as C++
print("Last element:",   nodes[-1]) # negative index = from end
print("Slice [1:3]:",    nodes[1:3])# slicing: up to but NOT including index 3

In [ ]:
# ── 1.3  Loops ───────────────────────────────────────────────────────────────
# C++: for(int i=0; i<5; i++) { ... }
# Python:
print("=== for loop ===")
for i in range(5):
    print(f"  node {i}")

print("\n=== loop over a list ===")
edges = [(0,3), (0,4), (1,3), (1,4), (2,3), (2,4)]  # 5-node Max-Cut edges
for u, v in edges:
    print(f"  edge ({u}—{v})")

In [ ]:
# ── 1.4  Functions ───────────────────────────────────────────────────────────
# C++: int add(int a, int b) { return a + b; }
# Python:
def add(a, b):
    return a + b

# Functions can return multiple values (returns a tuple)
def min_max(lst):
    return min(lst), max(lst)

print(add(3, 4))
lo, hi = min_max([5, 1, 8, 2])
print(f"min={lo}, max={hi}")

In [ ]:
# ── 1.5  Dictionaries (key-value store) ──────────────────────────────────────
# We'll see measurement results as dictionaries later, e.g. {'00011': 448, '11100': 399}
counts = {'00011': 448, '11100': 399, '00000': 24}

print("All keys:   ", list(counts.keys()))
print("All values: ", list(counts.values()))
print("Count for '00011':", counts['00011'])

# Loop over dictionary
for bitstring, count in counts.items():
    print(f"  {bitstring}: {count} shots")

### ✏️ Exercise 1.1 — Write a `cut_value` function

For the 5-node Max-Cut graph with edges `(0,3),(0,4),(1,3),(1,4),(2,3),(2,4)`,
write a function `cut_value(bitstring, edges)` that:
- Takes a bitstring like `'00011'` (node 3 and 4 in group 1, rest in group 0)
- Returns how many edges cross between the two groups

An edge `(u,v)` is **cut** when `bitstring[u] != bitstring[v]`.

In [ ]:
def cut_value(bitstring, edges):
    """
    Count how many edges are cut by the given bitstring partition.
    bitstring: e.g. '00011'  (character at index i = group of node i)
    edges:     list of (u, v) tuples
    """
    # YOUR CODE HERE
    count = 0
    for u, v in edges:
        if bitstring[u] != bitstring[v]:
            count += 1
    return count


# ── Test your function ────────────────────────────────────────────────────────
edges = [(0,3),(0,4),(1,3),(1,4),(2,3),(2,4)]

test_cases = [
    ('00011', 6),   # optimal solution
    ('11100', 6),   # also optimal (complement)
    ('00000', 0),   # all same group → no cuts
    ('01001', 3),   # partial cut
]

print("Testing cut_value():")
all_pass = True
for bs, expected in test_cases:
    result = cut_value(bs, edges)
    status = '✓ PASS' if result == expected else f'✗ FAIL (got {result})'
    print(f"  cut('{bs}') = {result}  expected {expected}  {status}")
    if result != expected:
        all_pass = False

print("\nAll tests passed!" if all_pass else "\nSome tests failed — check your logic.")

---
## Part 2: NumPy — Vectors and Matrices for Quantum Computing

Quantum states are **vectors**, and quantum gates are **matrices**. NumPy makes this easy.

In [ ]:
import numpy as np

# ── 2.1  Quantum basis states as vectors ─────────────────────────────────────
ket_0 = np.array([1, 0])   # |0⟩
ket_1 = np.array([0, 1])   # |1⟩

print("|0⟩ =", ket_0)
print("|1⟩ =", ket_1)

In [ ]:
# ── 2.2  Quantum gates as matrices ───────────────────────────────────────────
# Hadamard gate H
H = (1/np.sqrt(2)) * np.array([[1,  1],
                                [1, -1]])

# Pauli-Z gate
Z = np.array([[1,  0],
              [0, -1]])

# Pauli-X gate (quantum NOT)
X = np.array([[0, 1],
              [1, 0]])

print("H =\n", np.round(H, 4))
print("\nZ =\n", Z)
print("\nX =\n", X)

In [ ]:
# ── 2.3  Applying a gate: matrix-vector multiplication ────────────────────────
# Gate application: |ψ'⟩ = H |0⟩
# In NumPy: H @ ket_0

psi_after_H = H @ ket_0
print("H|0⟩ =", psi_after_H)   # should be [1/√2, 1/√2]

psi_after_Z_on_1 = Z @ ket_1
print("Z|1⟩ =", psi_after_Z_on_1)  # should be [0, -1] = -|1⟩

# Chaining gates: apply H then Z
result = Z @ (H @ ket_0)
print("Z H|0⟩ =", np.round(result, 4))

In [ ]:
# ── 2.4  Tensor product (⊗) for two-qubit systems ────────────────────────────
# Two-qubit state |00⟩ = |0⟩ ⊗ |0⟩
ket_00 = np.kron(ket_0, ket_0)
ket_01 = np.kron(ket_0, ket_1)
ket_10 = np.kron(ket_1, ket_0)
ket_11 = np.kron(ket_1, ket_1)

print("|00⟩ =", ket_00)  # [1, 0, 0, 0]
print("|01⟩ =", ket_01)  # [0, 1, 0, 0]
print("|10⟩ =", ket_10)  # [0, 0, 1, 0]
print("|11⟩ =", ket_11)  # [0, 0, 0, 1]

# Two-qubit H⊗H gate
HH = np.kron(H, H)
print("\nH⊗H applied to |00⟩:")
print(np.round(HH @ ket_00, 4))  # equal superposition of all 4 states

### ✏️ Exercise 1.2 — Manual gate application

Using NumPy, compute the following and print the results:
1. `X|0⟩`  — what state do you get?
2. `H|1⟩`  — what state do you get?
3. `Z(H|0⟩)` — first apply H to |0⟩, then Z to the result. What is the output?
4. What is the **probability** of measuring |0⟩ in the state H|0⟩?
   (Hint: probability = amplitude²)

In [ ]:
# YOUR CODE HERE

# 1. X|0⟩
print("1. X|0⟩ =", X @ ket_0)

# 2. H|1⟩
print("2. H|1⟩ =", np.round(H @ ket_1, 4))

# 3. Z(H|0⟩)
print("3. Z(H|0⟩) =", np.round(Z @ (H @ ket_0), 4))

# 4. Probability of |0⟩ in H|0⟩
state = H @ ket_0
prob_0 = state[0]**2
print(f"4. P(|0⟩) in H|0⟩ = {prob_0:.4f} = {prob_0} ≈ 0.5")

---
## Part 3: Your First Qiskit Circuit

Now we switch from manual matrix math to **Qiskit**, which builds and runs quantum circuits for us.

In [ ]:
# Install Qiskit if running on Google Colab
# (Skip if already installed in your local environment)
# !pip install qiskit qiskit-aer matplotlib -q

# Verify installation
import qiskit
print("Qiskit version:", qiskit.__version__)

In [ ]:
from qiskit import QuantumCircuit

# ── 3.1  Build a simple circuit ──────────────────────────────────────────────
# Create a circuit with 1 qubit and 1 classical bit
qc = QuantumCircuit(1, 1)

# All qubits start in |0⟩
# Apply Hadamard gate to qubit 0
qc.h(0)

# Measure qubit 0 → classical bit 0
qc.measure(0, 0)

# Draw the circuit
print(qc.draw('text'))

In [ ]:
from qiskit_aer import AerSimulator
from qiskit import transpile

# ── 3.2  Run on the Aer simulator ────────────────────────────────────────────
simulator = AerSimulator()

# Transpile (compile) the circuit for the simulator
compiled = transpile(qc, simulator)

# Run for 1000 'shots' (repeated measurements)
job    = simulator.run(compiled, shots=1000)
result = job.result()
counts = result.get_counts()

print("Measurement results (1000 shots):")
print(counts)
print("\nP('0') ≈", counts.get('0',0)/1000)
print("P('1') ≈", counts.get('1',0)/1000)

In [ ]:
import matplotlib.pyplot as plt

# ── 3.3  Plot results ────────────────────────────────────────────────────────
labels  = list(counts.keys())
values  = list(counts.values())

plt.figure(figsize=(4, 3))
plt.bar(labels, values, color=['steelblue','tomato'])
plt.xlabel('Measurement outcome')
plt.ylabel('Counts (out of 1000)')
plt.title('H|0⟩ measurement — should be ~50/50')
plt.tight_layout()
plt.show()

In [ ]:
from qiskit.quantum_info import Statevector

# ── 3.4  Inspect the statevector (no measurement needed) ─────────────────────
qc_no_measure = QuantumCircuit(1)
qc_no_measure.h(0)

sv = Statevector(qc_no_measure)
print("Statevector of H|0⟩:", sv)
print("Probabilities:        ", sv.probabilities_dict())

---
## Part 4: Multi-Qubit Circuits

Real quantum algorithms use many qubits. Let's build a 5-qubit superposition — the starting point of QAOA on our Max-Cut problem.

In [ ]:
# ── 4.1  5-qubit uniform superposition ───────────────────────────────────────
# This is Step 1 of QAOA: "Everyone on Stage"
n_qubits = 5
qc5 = QuantumCircuit(n_qubits, n_qubits)

# Apply H to all qubits
for i in range(n_qubits):
    qc5.h(i)

# Measure all
qc5.measure(range(n_qubits), range(n_qubits))

print(qc5.draw('text'))

In [ ]:
# ── 4.2  Run and verify uniform distribution ──────────────────────────────────
compiled5 = transpile(qc5, simulator)
result5   = simulator.run(compiled5, shots=3200).result()
counts5   = result5.get_counts()

print(f"Number of unique bitstrings observed: {len(counts5)} (expect ~32)")
print("\nSample of counts (should all be roughly 3200/32 ≈ 100):")
for bs, cnt in sorted(counts5.items())[:8]:
    print(f"  |{bs}⟩ : {cnt}")

In [ ]:
# ── 4.3  Plot cut values of all observed bitstrings ───────────────────────────
edges = [(0,3),(0,4),(1,3),(1,4),(2,3),(2,4)]

cut_counts = {}  # cut_value → total measurement count
for bs, cnt in counts5.items():
    cv = cut_value(bs, edges)
    cut_counts[cv] = cut_counts.get(cv, 0) + cnt

cv_sorted = sorted(cut_counts.keys())
plt.figure(figsize=(6, 3))
plt.bar(cv_sorted, [cut_counts[v] for v in cv_sorted], color='steelblue')
plt.xlabel('Cut value')
plt.ylabel('Total shots')
plt.title('Distribution of cut values after uniform superposition\n(before QAOA optimization — should be ~flat)')
plt.xticks(cv_sorted)
plt.tight_layout()
plt.show()
print("\nNotice: without QAOA, all cut values are equally likely.")
print("QAOA will amplify probability of cut = 6 (the maximum).")

---
### ✏️ Exercise 1.3 — Build a Bell State circuit

A **Bell state** is one of the simplest entangled two-qubit states:
$$|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$$

It is created by:
1. Applying H to qubit 0
2. Applying a CNOT gate with qubit 0 as control and qubit 1 as target (`qc.cx(0, 1)`)

**Tasks:**
1. Build this circuit and draw it
2. Run it for 1000 shots and print the counts
3. What do you observe? Why do you never see `'01'` or `'10'`?

In [ ]:
# YOUR CODE HERE
qc_bell = QuantumCircuit(2, 2)

# Step 1: Hadamard on qubit 0
qc_bell.h(0)

# Step 2: CNOT (control=0, target=1)
qc_bell.cx(0, 1)

# Measure both qubits
qc_bell.measure([0, 1], [0, 1])

print(qc_bell.draw('text'))

# Run
compiled_bell = transpile(qc_bell, simulator)
result_bell   = simulator.run(compiled_bell, shots=1000).result()
counts_bell   = result_bell.get_counts()
print("\nBell state measurement counts:", counts_bell)
print("""
Observation: Only '00' and '11' appear.
The two qubits are ENTANGLED — measuring one instantly determines the other.
This is the quantum resource that QAOA exploits for correlated qubit updates.
""")

---
## ✅ Lab 1 Summary

| Concept | What you learned |
|---------|------------------|
| Python basics | Variables, lists, loops, functions, dicts |
| NumPy | Vectors and matrices, `@` for multiplication, `np.kron` for tensor product |
| Qiskit | `QuantumCircuit`, gates (`h`, `cx`, `measure`), `AerSimulator`, `Statevector` |
| Quantum concepts | Superposition (H gate), measurement, entanglement (Bell state) |
| Max-Cut preview | 5-qubit uniform superposition = equal probability over all 32 partitions |

## 🔭 Preview of Lab 2
Next lab: **Qubits and the Bloch Sphere** — we'll visualize single-qubit states geometrically and explore how the Hadamard and Pauli gates move a qubit around the sphere.

---
*QOS Lab 1 | Prof. Chansu Yu | Cleveland State University*